In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('D:\Manoj_Honors\Merged_Data.csv')
df.isna().sum()

<>:1: SyntaxWarning: invalid escape sequence '\M'
<>:1: SyntaxWarning: invalid escape sequence '\M'
C:\Users\Faculty\AppData\Local\Temp\ipykernel_1166256\4235148299.py:1: SyntaxWarning: invalid escape sequence '\M'
  df = pd.read_csv('D:\Manoj_Honors\Merged_Data.csv')


Time               0
Station_ID         0
NO2           113000
PM2.5         111193
Ozone         137327
Lat                0
Lon                0
dtype: int64

In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.spatial.distance import cdist

# Load the CSV file
df = pd.read_csv(r'D:\Manoj_Honors\Merged_Data.csv')  # Replace 'your_file.csv' with the path to your CSV

# Inspect the data to check for columns and missing values
print(df.head())  # Display the first few rows

# Function to calculate IDW
def idw_imputation(df, target_column='NO2', power=2):
    # Coordinates of stations (Lat and Lon)
    coords = df[['Lat', 'Lon']].values
    # Identify missing values
    missing_idx = df[target_column].isnull()
    
    # List to store imputed values
    imputed_values = []

    for idx in range(len(df)):
        if missing_idx[idx]:
            # For missing value, calculate the weighted average of other values
            distances = cdist([coords[idx]], coords[~missing_idx], metric='euclidean').flatten()
            weights = 1 / (distances ** power)
            valid_values = df[target_column][~missing_idx]
            imputed_value = np.dot(weights, valid_values) / np.sum(weights)
            imputed_values.append(imputed_value)
        else:
            # For non-missing values, keep the original value
            imputed_values.append(df[target_column][idx])

    df[target_column] = imputed_values
    return df

# Impute missing values in the 'NO2' column using IDW
df_imputed = idw_imputation(df)

# Calculate the evaluation metrics
def evaluate_metrics(original_df, imputed_df, target_column='NO2'):
    # Metrics: R², RMSE, MAE, MSE, MAPE
    y_true = original_df[target_column].dropna()
    y_pred = imputed_df[target_column].dropna()

    # Calculate R²
    r2 = r2_score(y_true, y_pred)
    
    # Calculate RMSE
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # Calculate MAE
    mae = mean_absolute_error(y_true, y_pred)
    
    # Calculate MSE
    mse = mean_squared_error(y_true, y_pred)
    
    # Calculate MAPE
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    
    return r2, rmse, mae, mse, mape

# Evaluate the imputed data
r2, rmse, mae, mse, mape = evaluate_metrics(df, df_imputed)
print(f"R²: {r2}\nRMSE: {rmse}\nMAE: {mae}\nMSE: {mse}\nMAPE: {mape}%")

# Optionally, save the imputed dataset to a new CSV
df_imputed.to_csv('imputed_data.csv', index=False)


                  Time  Station_ID    NO2   PM2.5  Ozone        Lat        Lon
0  2023-01-01 00:00:00           0  30.90  134.00    3.3  28.815329  77.153010
1  2023-01-01 00:00:00           1  86.10  148.00    8.8  28.646835  77.316032
2  2023-01-01 00:00:00           2  39.00  147.00    2.4  28.695381  77.181665
3  2023-01-01 00:00:00           3  18.78   72.02    NaN  28.470691  77.109936
4  2023-01-01 00:00:00           4  13.50  185.00    8.2  28.776200  77.051074


C:\Users\Faculty\AppData\Local\Temp\ipykernel_1166256\2401850151.py:26: RuntimeWarning: divide by zero encountered in divide
  weights = 1 / (distances ** power)
C:\Users\Faculty\AppData\Local\Temp\ipykernel_1166256\2401850151.py:28: RuntimeWarning: invalid value encountered in scalar divide
  imputed_value = np.dot(weights, valid_values) / np.sum(weights)


R²: 1.0
RMSE: 0.0
MAE: 0.0
MSE: 0.0
MAPE: 0.0%
